### 10 - Building files to test dependency parsing
In this notebook, we will be build a function which can parse sentences with certain attributes (such as not containing unparsed readings, containing a locative noun, etc.) to more specifically test our dependency grammar.

In [2]:
from pathlib import Path

In [3]:
import csv
import ast
import string
import rich
from typing import Tuple

# Helper function to help skip punctuation readings
def is_punct(form: str) -> bool:
    return all(c in string.punctuation for c in form)

def extract_examples(
    csv_path: str | Path,
    *,
    limit: int = 100,
    ojibwe_out: str | Path,
    english_out: str | Path,
    using_patterns: bool = False,
    patterns: tuple[str, ...] = None,
) -> None:
    """
    Read csv file formatted with Ojibwe, English, and FST readings,
    find rows whose fst_readings contains no missing analyses and 
    at least one analysis with a tag from PATTERNS (if set), and
    write up to 'limit' matching sentences into two parallel files.

    Each output file has one sentence per line, corresponding lines align.
    """
    picked = 0
    seen: set[Tuple[str, str]] = set()        # avoid duplicates

    with (
        open(csv_path, newline="", encoding="utf-8") as fin,
        open(ojibwe_out,  "w", encoding="utf-8") as f_oj,
        open(english_out, "w", encoding="utf-8") as f_en,
    ):
        reader = csv.DictReader(fin)

        for row in reader:
            if picked >= limit:
                break

            # 1. Recover analyses list
            try:
                analyses = ast.literal_eval(row["fst_readings"])
            except Exception:
                continue                     

            # 2. Skip rows with unparsed tokens excluding punctuation
            if any(
                (not tok.get("fst_analyses"))
                and (not is_punct(tok.get("word_form", "")))
                for tok in analyses
            ):
                continue
            
            # 3. Require at least one tag in patterns (if set)
            if using_patterns: 
                if not patterns:
                    return rich.print(f"[bold red] Error: Patterns argument not specified")
                
                pattern_found = any(
                    any(pat in ana for pat in patterns)
                    for tok in analyses
                    for ana in tok.get("fst_analyses", [])
                )                   
                if not pattern_found:
                   continue

            # 4. Deduplicate sentence pairs
            oj_sent = row["ojibwe"].strip()
            en_sent = row["english"].strip()
            pair    = (oj_sent, en_sent)
            if pair in seen:
                continue          # skip duplicates
            seen.add(pair)

            # 5. Write to files
            f_oj.write(oj_sent + "\n")
            f_en.write(en_sent + "\n")
            picked += 1

<H4>Example usage:

Set paths:

In [4]:
AMBIGUOUS_SENTS_PATH = "../data/parallel_data/raw/sentences_with_ambiguity.csv"
OJIBWE_OUT_PATH = "../data/parallel_data/treebank_sentences/ojibwe_100.txt"
ENGLISH_OUT_PATH = "../data/parallel_data/treebank_sentences/english_100.txt"

Call function:

In [6]:
# modify or add new tuples to check for different patterns
LOC_PATTERNS: tuple[str, ...] = (
    "ADVLoc",        # adverb locative tag
    "+Loc",          # locative suffix in FST string
    " Loc",          # space-locative
)

extract_examples(csv_path=AMBIGUOUS_SENTS_PATH, ojibwe_out=OJIBWE_OUT_PATH, english_out=ENGLISH_OUT_PATH, using_patterns=False)
